# Version 2.1

- (2026.5.10) Add independent embedding python (work both Windows and Linux)
- (2026.5.10) Fix potential error in `ingest_embeddings`. Add global cache variable to store important content
- (2026.5.12) Improve `ingest_embeddings`: add cache feature which can restore the batch after unexpected crash happening
- (2026.5.12) Single large embedding database may choke the system when trainning. Therefore, we split it to multiple small parts. And use `MergerRetriever` to combine them. For example:
```text
embedding_db/
├── wqb_forum_china_embedding_db/
├── wqb_forum_global_embedding_db/
├── wqb_forum_research_embedding_db/
├── wqb_forum_tips_embedding_db/
└── wqb_official_docs_embedding_db/
```


In [ ]:
# Temp install command
# %pip install langchain langchain-text-splitters langchain-chroma langchain-openai langchain-community langchain-huggingface langchain-classic crewai crewai-tools sentence-transformers ipywidgets pypdf tqdm ansi2html

In [1]:
import os
import json
import requests
from pathlib import Path
from crewai import Agent, Task, Crew, Process, LLM
from langchain_chroma import Chroma
from langchain_classic.retrievers import MergerRetriever
from crewai.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from config.config import API_KEY_MOONSHOT
import logging
import datetime

In [2]:
def setup_logger(log_dir, log_name, logger_obj_name="logger_obj_name"):
    """
    Configure the logger system: output to both console and file simultaneously.
    """
    # 1. If the log directory doesn't exist, create it
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        print(f"Log directory created: {log_dir}")

    # 2. Generate a timestamped log filename, e.g., scraper_log_20231027_103000.log
    log_filename = f"{log_name}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    log_filepath = os.path.join(log_dir, log_filename)

    # 3. Create Logger object
    logger = logging.getLogger(logger_obj_name)
    logger.setLevel(logging.INFO) # Set the minimum logger level
    logger.handlers = [] # Clear previous handlers to prevent duplicate printing

    # --- Define a unified format (time accurate to the second) ---
    # %(asctime)s : Time
    # %(levelname)s : Log level (INFO/ERROR)
    # %(message)s : Your message content
    file_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )
    console_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )

    # --- Handler 1: File output (detailed, with timestamp) ---
    file_handler = logging.FileHandler(log_filepath, encoding='utf-8')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # --- Handler 2: Console output (concise, for human reading) ---
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    logger.info(f"✅ logger System Started")
    logger.info(f"Log file path: {log_filepath}")
    return logger

In [3]:
# ====================== CONFIG: EVERYTHING ON YOUR OTHER DRIVE ======================
BASE_DIR = Path.cwd() # current directory
# WQB_FORUM_PATH = BASE_DIR / "Docs" / "Forums"
WQB_FORUM_CHINA_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_china_consultant_pdf"
WQB_FORUM_GLOBAL_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_global_consultant_pdf"
WQB_FORUM_RESEARCH_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_research_pdf"
WQB_FORUM_TIPS_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_brain_tips_pdf"
WQB_OFFICIAL_DOCS_PATH = BASE_DIR / "Docs" / "OfficialDocs"
OPERATOR_FILE_PATH = BASE_DIR / "Operators" / "Operators-Agent.json"
DATAFIELDS_FILE_PATH = BASE_DIR / "DataFields" / "Datafield-Dataset-Category-Description.json"
# Note: PaymentPolicy store in WQB_FORUM_TIPS_PATH since it has only few pdfs
# WQB_PAYMENT_POLICY_PATH = BASE_DIR / "Docs" / "PaymentPolicy"

EMBEDDING_DB_FORUM_CHINA_DIR = BASE_DIR / "embedding_db" / "wqb_forum_china_embedding_db"
EMBEDDING_DB_FORUM_GLOBAL_DIR = BASE_DIR / "embedding_db" / "wqb_forum_global_embedding_db"
EMBEDDING_DB_FORUM_RESEARCH_DIR = BASE_DIR / "embedding_db" / "wqb_forum_research_embedding_db"
EMBEDDING_DB_FORUM_TIPS_DIR = BASE_DIR / "embedding_db" / "wqb_forum_tips_embedding_db"
EMBEDDING_DB_OFFICIALDOCS_DIR = BASE_DIR / "embedding_db" / "wqb_official_docs_embedding_db"
EMBEDDING_DB_DIRECTORIES = {
    EMBEDDING_DB_FORUM_CHINA_DIR,
    EMBEDDING_DB_FORUM_GLOBAL_DIR,
    EMBEDDING_DB_FORUM_RESEARCH_DIR,
    EMBEDDING_DB_FORUM_TIPS_DIR,
    EMBEDDING_DB_OFFICIALDOCS_DIR
}
HF_CACHE_DIR = BASE_DIR / "cache" / "hf"
LOG_DIR = BASE_DIR / "logs" / datetime.datetime.now().strftime("%Y%m")

# Create the folders (pathlib makes this easy too)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_CHINA_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_RESEARCH_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_TIPS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_OFFICIALDOCS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ====================== INITIALIZE LOGGER ======================
logger = setup_logger(LOG_DIR, "wqb_agent", "wqb_main_logger")

# ====================== LOAD JSON DATA ======================
with open(OPERATOR_FILE_PATH, 'r', encoding='utf-8') as f:
    operators_data = json.load(f)

with open(DATAFIELDS_FILE_PATH, 'r', encoding='utf-8') as f:
    datafields_data = json.load(f)

[26-5-12 16:47:06][INFO] ✅ logger System Started
[26-5-12 16:47:06][INFO] Log file path: d:\AI_Data\Computer\WorldQuantBrain-Agent\logs\202605\wqb_agent-20260512-164706.log


In [4]:
# Get Model List

base_url = "https://api.moonshot.cn/v1/models"
API_KEY = API_KEY_MOONSHOT

In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}"}
response = requests.get(base_url, headers=headers)
if response.status_code == 200:
    models = response.json()
    logger.info(f"Response: {models}")
    for model in models['data']:
        logger.info(f"Model ID: {model['id']}")
else:
    logger.error(f"Error: {response.status_code}, {response.text}")

[26-5-12 15:06:53][INFO] Response: {'object': 'list', 'data': [{'created': 1778239338, 'id': 'kimi-k2-0711-preview', 'object': 'model', 'owned_by': 'moonshot', 'permission': [{'created': 0, 'id': '', 'object': '', 'organization': 'moonshot', 'group': 'moonshot', 'is_blocking': False}], 'root': '', 'parent': '', 'context_length': 131072}, {'created': 1778239338, 'id': 'kimi-k2.5', 'object': 'model', 'owned_by': 'moonshot', 'permission': [{'created': 0, 'id': '', 'object': '', 'organization': 'moonshot', 'group': 'moonshot', 'is_blocking': False}], 'root': '', 'parent': '', 'supports_image_in': True, 'supports_video_in': True, 'supports_reasoning': True, 'context_length': 262144}, {'created': 1778239338, 'id': 'moonshot-v1-32k', 'object': 'model', 'owned_by': 'moonshot', 'permission': [{'created': 0, 'id': '', 'object': '', 'organization': 'moonshot', 'group': 'moonshot', 'is_blocking': False}], 'root': '', 'parent': '', 'context_length': 32768}, {'created': 1778239338, 'id': 'moonshot-v

In [5]:
# Use your exact proxy settings
pro_model = "kimi-k2.5" # gemini-3.1-pro
flash_model = "moonshot-v1-128k" # gemini-3.0-flash-thinking

llm_gemini_pro = LLM(
    model=pro_model,   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=API_KEY,
    temperature=0.4,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

llm_gemini_flash = LLM(
    model=flash_model,   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=API_KEY,
    temperature=0.6,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

### Embedding (Multiple Database)

- Forums
- Official Docs
- Payment Policy

See `wqbagent_embedding.ipynb`

In [6]:
# ==================================== Initialize Retriever (for querying) ====================================
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # excellent for Chinese
    cache_folder=str(HF_CACHE_DIR),  # Use the custom cache directory
    model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
    encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
    show_progress=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
# Old single database
# vectorstore = Chroma(persist_directory=EMBEDDING_DB_FORUM_GLOBAL_DIR, embedding_function=embeddings)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# 2. Initialize a list to hold individual retrievers
retrievers = []

for db_dir in EMBEDDING_DB_DIRECTORIES:
    # Load the vectorstore for each directory
    vectorstore = Chroma(persist_directory=db_dir, embedding_function=embeddings)
    
    # Create a retriever for it
    # Note: If you set k=8 here, each DB will return 8 docs. 
    # With 5 databases, you'll get up to 40 documents back initially.
    retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
    retrievers.append(retriever)

# 3. Combine all individual retrievers into one
combined_retriever = MergerRetriever(retrievers=retrievers)

# Now you can use this exactly like a standard single retriever
combined_retriever.invoke("alpha")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(id='13395bda63a7606d01b61085706897d6b95e6dbc5aec803667fa6bdfe95e4eac', metadata={'creationdate': '2026-05-04T19:16:37+00:00', 'moddate': '2026-05-04T19:16:37+00:00', 'source': '/data/Docs/Forums/wqb_global_consultant_pdf/39968818648215-2026-04-23-What Actually Improves Alpha Performance_.pdf', 'page': 0, 'title': 'What Actually Improves Alpha Performance? – WorldQuant BRAIN', 'total_pages': 2, 'page_label': '1', 'producer': 'Skia/PDF m141', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36', 'source_type': 'forum_global_pdf'}, page_content="WorldQ uant BRAIN > Community> Global Consultant Community for Staying Ahead\nWhat Actually Improves Alpha Performance?\nJM47610\n10 days ago\nAlpha performance to me isn’t about chasing signals.\nIt’s about controlling behavior\xa0across time, regimes, and combinations.The edge is rarely in complexity. It’s in\nclarity, discipline, and diversification.\nW hat helps 

### Search Tool

In [16]:
# ====================== DOCS SEARCH TOOL ======================
@tool("retrieve_text_data")
def retrieve_text_data(query: str) -> str:
    """Fetches relevant text snippets based on a query.
    Input a highly specific financial concept or math operator (e.g., 'supply chain momentum', 'analyst revision').
    Returns text context to be used for answering user queries."""
    
    docs = retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ====================== JSON SEARCH TOOLS ======================
@tool("search_operators")
def search_operators(query: str) -> str:
    """Search the operator definitions and syntax. 
    Input a concept like 'neutralize', 'moving average', or 'rank'."""
    results = []
    query_lower = query.lower()
    for op_name, op_details in operators_data.items():
        if (query_lower in op_name.lower() or 
            query_lower in op_details.get('description', '').lower() or 
            query_lower in op_details.get('category', '').lower()):
            
            res = f"Operator: {op_name}\nSyntax: {op_details.get('definition')}\nDesc: {op_details.get('description')}"
            results.append(res)
            
        if len(results) >= 10:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching operators found."

@tool("search_datafields")
def search_datafields(query: str) -> str:
    """Search the data dictionary for dataset fields. 
    Input a concept like 'pollution', 'inventory', 'analyst', or 'price'."""
    results = []
    query_lower = query.lower()
    for field_name, field_details in datafields_data.items():
        if (query_lower in field_name.lower() or 
            query_lower in field_details.get('description', '').lower() or 
            query_lower in field_details.get('category_name', '').lower()): # Note: matching your JSON typo 'category_name'
            
            res = f"Field: {field_name}\nType: {field_details.get('type')}\nDesc: {field_details.get('description')}"
            results.append(res)
            
        if len(results) >= 15:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching data fields found."


# ====================== WQB SIMULATE API TOOL ======================
WQB_SIMULATE_API_URL = os.getenv("WQB_SIMULATE_API_URL", "").strip()
WQB_SIMULATE_TIMEOUT_SECONDS = int(os.getenv("WQB_SIMULATE_TIMEOUT_SECONDS", "120"))


@tool("wqb_simulate_api")
def wqb_simulate_api(settings: str, regular_formula: str) -> str:
    """Run alpha simulation by calling your WQB simulate API.

    Input:
    - settings: JSON string of simulator settings.
    - regular_formula: WorldQuant regular formula string.
    """
    if not WQB_SIMULATE_API_URL:
        return (
            "Simulation API URL is not configured. "
            "Set environment variable WQB_SIMULATE_API_URL first."
        )

    try:
        settings_payload = json.loads(settings) if isinstance(settings, str) else settings
    except json.JSONDecodeError as e:
        return f"Invalid settings JSON: {e}"

    payload = {
        "settings": settings_payload,
        "regular_formula": regular_formula,
    }

    try:
        response = requests.post(
            WQB_SIMULATE_API_URL,
            json=payload,
            timeout=WQB_SIMULATE_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        try:
            return json.dumps(response.json(), ensure_ascii=False)
        except Exception:
            return response.text
    except Exception as e:
        return f"Simulation API request failed: {e}"

In [ ]:
# ==================================== DEBUG: Test if your docs PDFs are loaded ====================================
print("Testing retrieve_text_data tool...")

test_result = retrieve_text_data.run("AllRightsReserved")

print("\n" + "="*80)
print("TOOL TEST RESULT:")
print(test_result)  # first 1500 chars
print("\n" + "="*80)
print(f"Length of returned text: {len(test_result)} characters")

In [ ]:
# ==================================== DEBUG: Test JSON search tools ====================================
print("Testing search_operators tool...")
test_op_result = search_operators.run("neutralize")
print("\n" + "="*80)
print("Search_Operators TOOL TEST RESULT:")
print(test_op_result)
print("\n" + "="*80)

In [ ]:
# ==================================== DEBUG: Test Datafields search tools ====================================
print("Testing search_datafields tool...")
test_df_result = search_datafields.run("health")
print("\n" + "="*80)
print("Search_DataFields TOOL TEST RESULT:")
print(test_df_result)
print("\n" + "="*80)

### Test Agents (Test whether the search tool is usable)

In [ ]:
from wqbquant_searchtool_test import test_agents
test_agents(retrieve_text_data, search_operators, search_datafields, llm_gemini_flash)

### Agents & Crew

In [ ]:
# ====================== AGENTS (Your Quant Research Team) ======================
researcher = Agent(
    role="WorldQuant Docs Researcher & Master Analyst",
    goal="Use the `retrieve_text_data` tool to fetch context for the user's request. Base your output on the tool's results. Never answer from general knowledge.",
    backstory="""You are a veteran WorldQuant Brain consultant. You are an advanced AI agent equipped with a local vector database interface.
    You do not have the PDFs in your internal memory; you rely ENTIRELY on the `retrieve_text_data` tool. You use lateral thinking. 
    If a user asks about 'volume', you search for 'liquidity shock', 'turnover spike', or 'institutional block trades'.
    You always call the `retrieve_text_data` tool multiple times with completely different vocabulary each time to get a full picture.
    """,
    tools=[retrieve_text_data],
    llm=llm_gemini_pro,
    verbose=True,
    allow_delegation=False
)

ideator = Agent(
    role="Low-Correlation BRAIN Innovator",
    goal="Create truly innovative, submittable alphas using specific alternative data fields.",
    backstory="""You are a contrarian quant. You MUST use the `search_datafields` tool to find real, specific dataset names (like 'anti_pollution_policy_industry_rank') to build your hypotheses. Do not hallucinate data field names.""",
    tools=[search_datafields], # <--- ADDED TOOL
    llm=llm_gemini_pro,
    verbose=True
)

coder = Agent(
    role="WorldQuant BRAIN Expression Expert",
    goal="Convert the idea into a valid expression using exact Operator syntax and Exact Data fields.",
    backstory="""You are an ex-WorldQuant Brain coder. 
    1. You MUST use `search_operators` to check the exact syntax of functions (e.g., checking if add() takes a filter argument). 
    2. You MUST use `search_datafields` to ensure you are using the exact string name for data sets.
    You never guess operator syntax.""",
    tools=[search_operators, search_datafields], # <--- ADDED TOOLS
    llm=llm_gemini_flash,
    verbose=True
)

validator = Agent(
    role="WorldQuant Submission Validator",
    goal="Ensure the alpha is innovative, low-correlation, and ready to submit. Output ONLY in the exact user-specified format.",
    backstory="You are the final gatekeeper. You check for simulator compatibility, low correlation, and economic soundness. You never add extra text outside the required format.",
    tools=[wqb_simulate_api],
    llm=llm_gemini_pro,
    verbose=True
)

# ====================== TASKS & CREW ======================
task1 = Task(
    description="""
    Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.
    
    STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.
    STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor, call it again with a different phrase.
    STEP 3: Synthesize the returned database snippets.
    
    Focus on extracting real opinions and specific discussions.
    """,
    expected_output="Structured summary of the retrieved database content with direct quotes.",
    agent=researcher
)

task2 = Task(
    description="""
    Generate 3-5 genuinely innovative alpha ideas.
    CRITICAL: You MUST use the `search_datafields` tool to search for keywords related to your ideas (e.g., "ESG", "Analyst", "Supply Chain") and include the EXACT field names in your output.
    Focus on low correlation and economic rationale.
    """,
    expected_output="Numbered list of 3-5 alpha ideas containing exact Data Field names, hypothesis, and low-correlation justification.",
    agent=ideator
)

task3 = Task(
    description="""
    Take the BEST idea from Task 2 and write a clean, valid WorldQuant BRAIN expression.
    CRITICAL: You MUST use the `search_operators` tool to verify the syntax of every math/logic function you plan to use before writing the final expression.
    Choose realistic Target Settings (Region, Universe, Neutralization, Delay, Decay, Truncation).
    """,
    expected_output="One complete alpha in the exact user format (Alpha Name + Economic Hypothesis + Target Settings + Full BRAIN Expression).",
    agent=coder
)

task4 = Task(
    description="""
    Act as strict WorldQuant reviewer.
    Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic sense.
    If needed, improve it slightly.

    CRITICAL: Before finalizing, call `wqb_simulate_api` exactly once with:
    - settings: a JSON string that includes Region, Universe, Neutralization, Delay, Decay, and Truncation.
    - regular_formula: the final formula string.

    Use simulation output to decide whether the alpha is good enough; if not, improve the formula once and return the improved one.

    THEN output ONLY the final alpha in the EXACT format the user wants:
    
    **Alpha Name:** ...
    **Economic Hypothesis:** ...
    **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ | Truncation: ___
    **Full BRAIN Expression:** ...
    
    Do not add any extra explanation or text outside this format.
    """,
    expected_output="Final alpha in the exact markdown format requested by the user.",
    agent=validator
)

crew = Crew(
    agents=[researcher, ideator, coder, validator],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,
    verbose=True,
    # tracing=True
)

In [ ]:
# ====================== RUN ======================
if __name__ == "__main__":
    result = crew.kickoff(inputs={
    "user_request": """
    Explore forum discussions specifically around Analyst Estimates (EPS, Revisions) or Supply Chain inventory data. 
    Find out what basic combinations people are using, and generate alphas that apply non-linear operators 
    (like sign, abs, or conditional logic) to these specific datasets to achieve low correlation.
    """.strip()
    })
    print(result)